# M6 — Riyadh Travel Planner / مخطّط الرياض

**التجارب الحية متوقفة افتراضيًا.** شغّلي الإعدادات أولًا واقرئي المخرجات. لا تحتاج الفحوص المحلية إلى مفتاح.

This notebook implements the approved shared policy. A pilot is separate from the formal study. Mock fixtures are not research results. Upload `M6_Submission.zip` only if the source is not already present. Download evidence after every live chunk.

Completed study: formal-v3 (9 November 2026), 120 live records. Read reports/M6_Report.md and m6_runs/formal-v3/analysis. This notebook retains the original pilot/formal-v1 workflow as execution history. Do not rerun earlier cohorts or overwrite interrupted evidence. The Drive recovery controllers are scripts/M6_Run_Persistent.py and scripts/M6_Run_Resumed.py.


In [ ]:
# Safe defaults: change a flag only after approval for that specific action.
INSTALL_DEPENDENCIES = False
UPLOAD_REPOSITORY_ZIP = False
APPROVE_LIVE_PILOT = False
APPROVE_FORMAL_CHUNK = False
PILOT_VERSION = "pilot-v1"
FORMAL_VERSION = "formal-v1"
PILOT_NEW_RECORD_CAP = 2
PILOT_HTTP_CAP = 60
FORMAL_NEW_RECORD_CAP = 6
FORMAL_HTTP_CAP = 180


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

if UPLOAD_REPOSITORY_ZIP:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one repository ZIP")
    target = Path("/content/m6_project").resolve()
    target.mkdir(parents=True, exist_ok=True)
    archive = next(iter(uploaded))
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            destination = (target / name).resolve()
            if target not in destination.parents and destination != target:
                raise ValueError("Unsafe ZIP member")
            if destination.exists() and not name.endswith("/"):
                raise ValueError("Existing files are preserved; use a fresh upload directory")
        z.extractall(target)

PROJECT = Path.cwd()
if not (PROJECT / "m6planner").is_dir():
    candidate = Path("/content/m6_project/M6_Submission")
    if candidate.is_dir():
        PROJECT = candidate
READY = (PROJECT / "m6planner").is_dir()
if READY:
    os.chdir(PROJECT)
    if str(PROJECT) not in sys.path:
        sys.path.insert(0, str(PROJECT))
    print("Project ready:", PROJECT)
else:
    print("Safe stop: upload the M6 source ZIP or open this notebook from its project folder.")

if INSTALL_DEPENDENCIES:
    if not READY:
        raise RuntimeError("Source folder must be ready before installation")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-gemini.txt"])


## فحص الإعداد / Safe inspection
This cell only imports local code and displays the frozen policy. It makes no provider request. The command-line runner also checks its own live approval flag before reading the credential or constructing a client.


In [ ]:
if READY:
    from dataclasses import asdict
    from m6planner.policy import Policy
    print(asdict(Policy()))
    print("Live execution remains disabled unless the separate approval flags are enabled.")


## التشغيل بعد الموافقة فقط / Approved live execution only
Pilot: G01/seed11, baseline and treatment, at most 2 new records and 60 HTTP requests. Start here after approval. The live protocol is the same for both conditions. A successful pilot is not a substitute for the formal cohort.

Formal: unchanged 20 cases × 3 seeds × 2 conditions. Each approved chunk starts at most 6 records and has an aggregate cap of 180 HTTP requests. Keep the same version and source to resume. Failed complete records are retained and skipped on resume. An interrupted incomplete record requires inspection; do not delete or silently rerun it.

Do not enable pilot and formal flags together. The secret is read only when a live flag is true. No automatic billing upgrade or model substitution is performed.


In [ ]:
if APPROVE_LIVE_PILOT and APPROVE_FORMAL_CHUNK:
    raise ValueError("Choose one separately approved action at a time")

if APPROVE_LIVE_PILOT or APPROVE_FORMAL_CHUNK:
    if not READY:
        raise RuntimeError("Project source is missing")
    from google.colab import userdata
    from m6planner.__main__ import main
    # Never display the secret or persist it in a notebook cell/output.
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    try:
        if APPROVE_LIVE_PILOT:
            main(["live", "--approve-live", "--kind", "pilot", "--version", PILOT_VERSION,
                  "--max-new-records", str(PILOT_NEW_RECORD_CAP), "--http-cap", str(PILOT_HTTP_CAP)])
        else:
            main(["live", "--approve-live", "--kind", "formal", "--version", FORMAL_VERSION,
                  "--max-new-records", str(FORMAL_NEW_RECORD_CAP), "--http-cap", str(FORMAL_HTTP_CAP)])
    finally:
        os.environ.pop("GEMINI_API_KEY", None)
else:
    print("No Gemini request made. Await approval for the paired pilot or formal chunk.")


## حفظ الأدلة وتحليلها / Preserve evidence and analyze
After a live chunk, download its entire version folder from `m6_runs/`, including incomplete journals if the session was interrupted. Do not overwrite prior versions. All output folders contain private project data.

After every expected formal record exists, run locally or in Colab:

```python
from m6planner.analysis import export_comparison
summary = export_comparison("m6_runs/formal-v1")
```

This offline analysis refuses partial cohorts and retains failed runs. Results go into `analysis/M6_Results.csv` and `analysis/M6_Comparison.json`. Independently verify with `python scripts/verify_m6.py m6_runs/formal-v1` before updating the report. Missing currency cost remains unknown; mock token/time values must never enter the experimental-results table.
